# NPUWattch hands-on — 1. PyTorchSim: a TPU-like NPU running one matmul

One PyTorchSim run: a 1024×1024×1024 float32 `torch.matmul` on a TPUv3-like
NPU with two 128×128 systolic arrays. This is the output you get from the
PyTorchSim session this morning, and NPUWattch reads it as is.

```
pytorchsim/tpu_like_fp32/
├── config.yml                   the PyTorchSim configuration the run used
├── togsim_results/*.log         the TOGSim log: machine shape in the header, activity at the end
├── outputs/<hash>/m5out/stats.txt   gem5 instruction counts
├── outputs/<hash>/meta.txt, *.mlir  operand data types of the kernel
├── booksim2_config/             the NoC configuration
├── compound_components.yaml     the hardware model: compounds made of NPUWattch primitives
├── projection.yaml              PyTorchSim counters -> activity of the compound elements
├── user_components.yaml         cost of custom blocks (an example entry)
└── run
```

There is no architecture file. PyTorchSim sets up its machine at run time, so
NPUWattch rebuilds it from the log header and `config.yml`. The command in
`run` is:

```bash
npuwattch --harness pytorchsim --togsim-dir togsim_results/ --gem5-dir outputs/ \
          --config-yml config.yml --booksim-dir booksim2_config/ \
          --node 7nm --tree --report out/
```

From a run root of your own, `NPUWattch/run.sh <root>` builds this command for
you.


In [1]:
import sys; sys.path.insert(0, ".")
from tutorial_helpers import *
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)
print("npuwattch:", npuwattch_binary())

npuwattch: /home/KNUEEhdd1/ys_knu/kmk/miniforge3/envs/npuwattch/bin/npuwattch


## 1. Run it, and read the rebuilt machine

In [2]:
ex32 = example("pytorchsim/tpu_like_fp32")
text32 = run(ex32)
show_tree(text32)

[INFO] Instance hierarchy (reconstructed from the run's model):


The tree is reconstructed from the run: one core with two arrays of
16 384 fp32 MACs and their weight registers, a 16 MB VMEM, 128 VPU lanes
(each with a scratchpad, a register file, an FPU, and an SFU), a DMA queue, a
32×32 NoC crossbar with its buffers, and 16 HBM channels. The PE type
(`fpmac`, 8-bit exponent, 23-bit mantissa) came from the kernel's MLIR: the
matmul takes `f32` inputs.

## 2. The messages

This run prints a few WARNING lines. Each one says what NPUWattch filled in
or extended for you.

In [3]:
show_messages(text32, kinds=("WARNING",))

[WARNING] vmem: capacity-only SRAM spec (134217728 bits): auto-applied macro template(s) 512x sram_256k grouped into banks of <= 16 macros; utilization 100.0%
[WARNING] vpu_spad: capacity-only SRAM spec (1048576 bits): auto-applied macro template(s) 4x sram_256k grouped into banks of <= 16 macros; utilization 100.0%
[WARNING] icnt_buf: capacity-only SRAM spec (65536 bits): auto-applied macro template(s) 1x sram_64k grouped into banks of <= 16 macros; utilization 100.0%
[WARNING] CustomMatMulwVpush: kernel-total events attributed per array in proportion to each array's active cycles
[WARNING] core0.vrf: width=256 is outside the characterized 8-128 for regfile — extrapolated


- `capacity-only SRAM spec ... auto-applied macro template(s)` (vmem,
  vpu_spad, icnt_buf): the configuration gives these memories as a capacity,
  not as a macro layout. NPUWattch builds each from characterized SRAM macros
  and reports the layout and utilization it chose.
- `CustomMatMulwVpush: kernel-total events attributed per array`: gem5 counts
  weight pushes once for the whole kernel. With two arrays, NPUWattch splits
  the count in proportion to each array's active cycles.
- `core0.vrf: width=256 is outside the characterized 8-128 ... extrapolated`:
  the vector register file is 256 bits wide. The register-file model extends
  beyond its characterized widths for it, and says so.

The INFO lines list what is deliberately left out of the totals: the scalar
core and its caches, the VCIX serializer, DRAM standby power, and the counters
that would double-count work already charged. The scope of the number is
always written down.

In [4]:
show_messages(text32, kinds=("INFO",))

[INFO] Harness mode: pytorchsim
[INFO] Instance hierarchy (reconstructed from the run's model):
[INFO] scalar core (with its L1 I-cache / D-cache, branch predictor and TLBs) is excluded from the energy model per PyTorchSim author guidance (2026-07-25, negligible area/power vs the systolic/vector datapath); gem5 stats are read only for numCycles and the instruction counts matching CustomMatMul(wVpush|iVpush|vpop)? and CustomV(exp|exp2|erf|tanh|sin|cos)
[INFO] CustomVlaneIdx instruction counts are deliberately not collected: it is lane-index generation, not an SFU op, and its energy is negligible
[INFO] BAR / MEMORY_BAR (barrier) instruction counters are not collected: synchronization is control-side work (scalar-core scope) and the stall time it causes is already inside total_exec_cycles, so it is charged as leakage over the window
[INFO] VCIX serializer/deserializer (the vector <-> systolic-array interface) is not modeled as a separate component: its push/pop events are already charged

## 3. Where the energy went

In [5]:
rep32 = report(ex32)
display(components_df(rep32, top=10))
show_totals(text32)
print("pJ/FLOP (DRAM included):", rep32["efficiency"]["pJ_per_flop_str"])

,component,class,model,instances,energy,share %,area
0,core0.array0.pe,fpmac,cal,16384,2.72e+03 µJ,44.3,483
1,core0.array1.pe,fpmac,cal,16384,2.72e+03 µJ,44.3,483
2,dram_chan,hbm,const,16,485 µJ,7.9,n/a
3,core0.array0.w_reg,regfile,cal,16384,76.8 µJ,1.3,15.4
4,core0.array1.w_reg,regfile,cal,16384,76.8 µJ,1.3,15.4
5,core0.vmem,sram,cal,1,16 µJ,0.3,3.56e+07
6,icnt_buf,sram,cal,32,17.3 µJ,0.3,1.74e+04
7,core0.vrf,regfile,cal,128,13.4 µJ,0.2,4.16e+03
8,core0.vpu_spad,sram,cal,128,4.9 µJ,0.1,2.78e+05
9,core0.dma_q,fifo,cal,1,8.34 µJ,0.1,2.38e+03


total energy = 6.132e+09 pJ (dyn 6.118e+09 + leak 1.468e+07); avg power = 1.108e+05 mW; exec = 5.534e-05 s
calibrated primitives available: fpadd, fpmul, fpmac, intadd, intmul, intmac, fpsfu, mxfpmac, fifo, regfile, simplemux, crossbar, fattree, foldedclos, sram
pJ/FLOP (DRAM included): 2.86 pJ/FLOP


Expected result at 7 nm: about **6.1 mJ** for the kernel, **2.9 pJ/FLOP**,
with the two systolic arrays at about 91% of the energy and DRAM at about 8%,
which is what a large matmul should look like.

## 4. Change the PE to bf16

A real TPU multiplies in bf16. `tpu_like_bf16/` is the same simulator run with
one file changed: in `compound_components.yaml`, the PE is a fixed bf16
`fpmac` instead of following the kernel's data type, and the weight register is
16 bits wide.

In [6]:
import difflib
a = (example("pytorchsim/tpu_like_fp32") / "compound_components.yaml").read_text().splitlines()
b = (example("pytorchsim/tpu_like_bf16") / "compound_components.yaml").read_text().splitlines()
print("\n".join(difflib.unified_diff(a, b, "fp32/compound_components.yaml", "bf16/compound_components.yaml", lineterm="", n=1)))

--- fp32/compound_components.yaml
+++ bf16/compound_components.yaml
@@ -42,4 +42,10 @@
 
+#
+# TPU-like bf16 variant: this design fixes the PE to a bf16 MAC. The PE does
+# not follow the kernel dtype. The PyTorchSim run is a float32 matmul, because
+# PyTorchSim cannot simulate bf16 or fp16. The systolic array does one MAC per
+# PE per cycle for each dtype, thus the cycle and MAC counts of that run apply.
+# The vpu below still follows the kernel dtype (fp32, as the TPU vector unit).
+
 systolic_mac:
-  select_primitive_by: dtype          # select the PE primitive from the kernel dtype
   elements:
@@ -47,4 +53,7 @@
     pe:                               # processing element = one MAC primitive
-      primitive: "{mac_primitive}"    # intmac / fpmac / mxfpmac
-      config: "{mac_config}"          # its config in NPUWattch names (for each kernel)
+      primitive: fpmac                # fixed: bf16 MAC, not the kernel dtype
+      config:
+        exponent_bits: 8              # bf16

In [7]:
ex16 = example("pytorchsim/tpu_like_bf16")
text16 = run(ex16)
rep16 = report(ex16)
pe = lambda rep: next(c for c in rep["components"] if c["name"].endswith("array0.pe"))
pd.DataFrame([
    {"model": "fp32 PE", "PE energy": pe(rep32)["energy_str"], "PE pJ/MAC": pe(rep32)["unit_energy_str"],
     "total": rep32["totals"]["energy_str"], "pJ/FLOP": rep32["efficiency"]["pJ_per_flop_str"], "area mm²": rep32["totals"]["area_mm2"]},
    {"model": "bf16 PE", "PE energy": pe(rep16)["energy_str"], "PE pJ/MAC": pe(rep16)["unit_energy_str"],
     "total": rep16["totals"]["energy_str"], "pJ/FLOP": rep16["efficiency"]["pJ_per_flop_str"], "area mm²": rep16["totals"]["area_mm2"]},
])

,model,PE energy,PE pJ/MAC,total,pJ/FLOP,area mm²
0,fp32 PE,2.72e+03 µJ,5.05,6.13e+03 µJ,2.86 pJ/FLOP,88.750
1,bf16 PE,837 µJ,1.56,2.29e+03 µJ,1.07 pJ/FLOP,77.842


The compound file says what the hardware *is*. Its partner, `projection.yaml`,
says which counter of the run drives each element and in which mode. This is
the entry for the systolic array:

In [8]:
src = (example("pytorchsim/tpu_like_fp32") / "projection.yaml").read_text().splitlines()
start = next(i for i, l in enumerate(src) if l.strip().startswith("systolic_mac:"))
print("\n".join(src[start:start + 12]))

  systolic_mac:

    CustomMatMul:                     # weight-stationary compute
      count_from:
        stat: systolic_active_cycles
        scale: "lanes*lanes"          # cycles × lanes² = MACs (Option A)
      elements:
        pe: hold_b                    # PE computes with b latched
        w_reg: idle                   # the weight stays, this action has no write

    CustomMatMulwVpush:               # weight load (push of a new weight)
      count_from:


Every active cycle of the array (`systolic_active_cycles`) is charged as
lanes² MAC operations in weight-stationary mode (`hold_b`), and the weight
registers sit idle during it. Notebook 4 writes a compound and a projection
from scratch.

Expected: the PE energy drops from about 5 pJ to about 1.6 pJ per MAC, and
the total from 6.1 mJ to 2.3 mJ (1.07 pJ/FLOP). DRAM and NoC traffic keep the
byte counts of the float32 run, as the bf16 README says.

This is the normal way to explore a hardware change with NPUWattch: keep the
simulator run, edit the hardware model in `compound_components.yaml`, run
again.

## 5. The native files

Add `-o native/` and NPUWattch writes the two files it built from the run: the
description and the activity table. They are the same two files you would
write by hand for a simulator that has no harness (Part 3, §2 of the basics).

In [9]:
_ = run(ex32, "-o", "native/", log_name="console_native.txt")
print("--- native/description.yaml (first 30 lines) ---")
show_file(ex32 / "native" / "description.yaml", max_lines=30)
print("...\n--- native/activity.csv ---")
show_file(ex32 / "native" / "activity.csv")

--- native/description.yaml (first 30 lines) ---
npuwattch:
  version: '1.0'
  technology:
    node: 7nm
    transistor: hp
    corner: TT
    voltage_offset_V: 0.0
    temperature_C: 25.0
  clock:
    frequency_MHz: 940.0
  components:
  - name: core0.array0.pe
    class: fpmac
    count: 16384
    attributes:
      number_format: fp
      data_width: 32
      exponent_bits: 8
      mantissa_bits: 23
      pipeline_stages: 4
  - name: core0.array0.w_reg
    class: register_file
    count: 16384
    attributes:
      data_width: 32
      mem_depth_per_bank: 1
      mem_banks: 1
      mem_r_ports: 1
      mem_w_ports: 1
  - name: core0.array1.pe
...
--- native/activity.csv ---
window,cycle_start,cycle_end,component,event,mode,count
0,0,52021,core0.array0.pe,op,hold_b,536870912
0,0,52021,core0.array1.pe,op,hold_b,536870912
0,0,52021,core0.array0.w_reg,idle,idle,536870912
0,0,52021,core0.array1.w_reg,idle,idle,536870912
0,0,52021,core0.array0.pe,idle,idle,1024
0,0,52021,core0.array1.pe,id

Run them directly, without the harness. The totals match.

In [10]:
text_native = run_npuwattch(["-d", "native/description.yaml", "-l", "native/activity.csv"], cwd=ex32)
show_totals(text_native)
print()
print("same total as the harness run:", abs(total_energy_pJ(text_native) - total_energy_pJ(text32)) / total_energy_pJ(text32) < 1e-6)

total energy = 6.132e+09 pJ (dyn 6.118e+09 + leak 1.468e+07); avg power = 1.108e+05 mW; exec = 5.534e-05 s
calibrated primitives available: fpadd, fpmul, fpmac, intadd, intmul, intmac, fpsfu, mxfpmac, fifo, regfile, simplemux, crossbar, fattree, foldedclos, sram

same total as the harness run: True


## 6. The report

In [11]:
show_report(ex32)

## Messages you will see, and what to do

**Every time.** The WARNING and INFO lines of section 2. None of them needs
action for this run.

**If something looks wrong:**

| What you see | What it means | What to do |
| --- | --- | --- |
| `run.sh: ... togsim_results/ not found` | The run root has no final TOGSim logs | Point at the root that contains `togsim_results/`; `outputs/<hash>/togsim_result/` holds autotune candidates, not the final run |
| `run.sh: both outputs/ and gem5_outputs/ present` | Two gem5 folders | Keep one, or pass `--gem5-dir` yourself |
| a WARNING that `vmem` is skipped | `config.yml` has no `core_spad_size_kb` | Add `core_spad_size_kb: <VMEM size in KB>` to `config.yml`; PyTorchSim ignores the key, NPUWattch uses it |
| `compound_components.yaml not found` | The definition files are not in the run root | Copy the three files from `pytorchsim/tpu_like_fp32/` into your run root |
| `the log header and config.yml disagree on <key>` | You passed a `config.yml` from another run | The log header wins; pass the matching file or drop `--config-yml` |

Next: `02_timeloop_eyeriss.ipynb`.
